In [2]:
# import necessary packages
import xarray as xr
import numpy as np
import pandas as pd
import os
import glob
import seaborn as sns
import matplotlib.pyplot as plt
sns.set(rc={'axes.facecolor': 'grey'})
plt.rcParams['figure.dpi'] = 300

In [3]:
os.chdir('C:/Users/edwin/OneDrive/Documents/GitHub/CHC')

In [4]:
# Initialize an empty dictionary to store DataFrames
dfs_dict = {}

list_of_files = glob.glob('data/csv/*.csv')
# Loop over all files
for f in list_of_files:
    # Generate the DataFrame
    df = pd.read_csv(f, sep=',', header=0, index_col=0)

    # Store the DataFrame in the dictionary with the year as key
    dfs_dict[f] = df

# Concatenate all DataFrames in the dictionary into one DataFrame
final_df = pd.concat(dfs_dict.values(), ignore_index=True)

# Convert the 'date_of_prediction' column to datetime format
final_df['date_of_prediction'] = pd.to_datetime(final_df['date_of_prediction'])
final_df['month_of_prediction'] = final_df['date_of_prediction'].dt.month
final_df = final_df.drop(columns=['date_of_prediction', 'realization_year'])
final_df['precip'] = final_df['precip']/30


In [ ]:
# Calculating all the statistics for each region and model

# groupby the region, model, season and month_of_prediction and calculate the corr between the predicted and actual precipitation for future use
corr = final_df.groupby(['region', 'model', 'season', 'month_of_prediction'])[['predicted_precip', 'precip']].corr(method = 'spearman').drop(['precip'], axis = 1).reset_index()
corr = corr.drop(corr.index[::2]).drop(columns = ['level_4'])
corr = corr.rename(columns = {'predicted_precip': 'corr'})

# Calculate mean and standard deviation
stat = final_df.groupby(['region', 'model', 'season', 'month_of_prediction']).agg(['mean', 'std']).reset_index()
stat.columns = ['region', 'model', 'season', 'month_of_prediction', 'pred_mean', 'pred_std', 'actual_mean', 'actual_std']

# Merging stat and spatial_means_corr to get 1 df with all values
stat_clean = stat.merge(corr, left_on=['region', 'model', 'season', 'month_of_prediction'], right_on=['region', 'model', 'season', 'month_of_prediction'], how='left').dropna()

# Calculating metrics
stat_clean['potential_skill'] = np.square(stat_clean['corr'])
stat_clean['conditional_bias'] = np.square(stat_clean['corr'] - (stat_clean['pred_std'] / stat_clean['actual_std']))
stat_clean['unconditional_bias'] = np.square((stat_clean['pred_mean'] - stat_clean['actual_mean']) / stat_clean['actual_std'])
stat_clean['skill_score'] = stat_clean['potential_skill'] - stat_clean['conditional_bias'] - stat_clean['unconditional_bias']

# dropping unnecessary columns
stat_clean = stat_clean.drop(['pred_mean','pred_std','actual_mean','actual_std'], axis = 1)



In [6]:
# Create a combined column for the region-season pair
stat_clean['region_season'] = stat_clean['region'] + " | " + stat_clean['season']

def draw_heatmap(*args, **kwargs):
    data = kwargs.pop('data')
    # Pivot so that x-axis is month_of_prediction and y-axis is model
    d = data.pivot(index='model', columns='month_of_prediction', values='potential_skill')
    
    # Define mapping of season to its first month
    season_to_first = {
        "MAM": 3,
        "AMJ": 4,
        "MJJ": 5,
        "FMA": 2
    }
    
    # Get the season for this facet (assumes all rows share the same season)
    season_val = data['season'].iloc[0]
    first_month = season_to_first.get(season_val)
    if first_month is not None:
        # Create a list of 7 months ending with the season's first month
        desired_order = [ ((first_month - 6 + i - 1) % 12) + 1 for i in range(7) ]
        # Filter to only the months present in the pivot
        new_order = [m for m in desired_order if m in d.columns]
        if new_order:
            d = d[new_order]
    
    sns.heatmap(d,
                vmin=0, vmax=1,
                cmap=sns.color_palette('Reds', 10),
                fmt=".2f",
                linewidths=0.1, linecolor='black',
                square=True)
    plt.xticks(fontsize=7)
    plt.yticks(fontsize=7)
    plt.gca().invert_xaxis()

# Facet using the combined region_season column with col_wrap to make layout cleaner.
fg = sns.FacetGrid(stat_clean, col='region_season', sharex=False, sharey=True, col_wrap=4)
fg.map_dataframe(draw_heatmap)
fg.set_titles("{col_name}")
fg.set_ylabels("Model")
fg.set_xlabels("Month Of Prediction")

# moving the title to the top of the figure
fg.fig.subplots_adjust(top=0.9)
fg.fig.suptitle('Potential Skill by Model and Month of Prediction', y=0.98)

plt.savefig('figures/seasonal_metrics/potential_skill.png')
plt.close()

In [ ]:
stat_clean.to

,region,model,season,month_of_prediction,corr,potential_skill,conditional_bias,unconditional_bias,skill_score,region_season
0,eastern_east_africa,CCSM4,MAM,1,0.260264,0.067737,3.587119e-03,2.125987,-2.061836,eastern_east_africa | MAM
1,eastern_east_africa,CCSM4,MAM,2,0.374633,0.140350,9.484789e-03,1.964059,-1.833193,eastern_east_africa | MAM
2,eastern_east_africa,CCSM4,MAM,3,0.614003,0.377000,1.170352e-07,1.296031,-0.919031,eastern_east_africa | MAM
3,eastern_east_africa,CCSM4,MAM,9,0.052419,0.002748,1.746199e-02,3.906009,-3.920723,eastern_east_africa | MAM
4,eastern_east_africa,CCSM4,MAM,10,-0.005499,0.000030,5.534380e-02,3.893003,-3.948317,eastern_east_africa | MAM
...,...,...,...,...,...,...,...,...,...,...
1285,west_africa,NCEP,JAS,3,0.488270,0.238407,2.443729e-01,0.119563,-0.125529,west_africa | JAS
1286,west_africa,NCEP,JAS,4,0.532991,0.284080,1.212464e-01,0.694241,-0.531408,west_africa | JAS
1287,west_africa,NCEP,JAS,5,0.394062,0.155285,2.007519e-01,2.874216,-2.919683,west_africa | JAS
1288,west_africa,NCEP,JAS,6,0.459677,0.211303,1.728182e-01,3.529483,-3.490997,west_africa | JAS


In [82]:
# keep first 3 months of prediction of each model for each region and season\
def keep_first_months_of_prediction(df):
    temp = df.copy()
    for season in temp['season'].unique():
        if (temp[temp['season'] == season]['month_of_prediction'] == 12).any() \
        & (temp[temp['season'] == season]['month_of_prediction'] == 1).any():
            temp.loc[(temp['season'] == season) & (temp['month_of_prediction'] >= 8),'month_of_prediction'] = \
            temp.loc[(temp['season'] == season) & (temp['month_of_prediction'] >= 8),'month_of_prediction'] - 12
        # keep 3 largest months of prediction of each model for each region and season
        max = temp[temp['season'] == season]['month_of_prediction'].max()
        temp.loc[(temp['season'] == season)] = \
        temp.loc[(temp['season'] == season) & (temp['month_of_prediction'] >= (max - 2))]
    return temp.dropna()


In [122]:
potential_skill = keep_first_months_of_prediction(stat_clean[stat_clean['model'] != 'MME'])
potential_skill = potential_skill.drop(columns = ['corr', 'conditional_bias', 'unconditional_bias', 'skill_score'])

# taking the mean of the month of prediction for each region and model and season
potential_skill = potential_skill.groupby(['region', 'model', 'season'])[['potential_skill']].mean().reset_index()
potential_skill

,region,model,season,potential_skill
0,eastern_east_africa,CCSM4,MAM,0.195029
1,eastern_east_africa,CCSM4,OND,0.249536
2,eastern_east_africa,CESM1,MAM,0.241240
3,eastern_east_africa,CESM1,OND,0.370867
4,eastern_east_africa,CMCC,MAM,0.262210
...,...,...,...,...
175,west_africa,GFDL,JAS,0.104363
176,west_africa,JMA,JAS,0.032314
177,west_africa,METEO,JAS,0.014781
178,west_africa,NASA,JAS,0.158595


In [123]:
an = pd.read_csv('data/csv/metrics/region_model_df_high.csv', sep=',', header=0, index_col=0)
bn = pd.read_csv('data/csv/metrics/region_model_df_low.csv', sep=',', header=0, index_col=0)
an = keep_first_months_of_prediction(an)
bn = keep_first_months_of_prediction(bn)
an = an.rename(columns = {'agreement': 'an_agreement'})
bn = bn.rename(columns = {'agreement': 'bn_agreement'})

# taking the month of prediction mean for each region and model and season
an = an[an['model'] != 'MME'].groupby(['region', 'model', 'season'])[['an_agreement']].mean().reset_index()
bn = bn[bn['model'] != 'MME'].groupby(['region', 'model', 'season'])[['bn_agreement']].mean().reset_index()

In [ ]:
# merge an, bn and potential_skill on region, model 
an_bn = an.merge(bn, left_on=['region', 'model', 'season'], right_on=['region', 'model', 'season'], how='left')
an_bn = an_bn.dropna()



In [155]:
# merge potential_skill with an_bn on region, model and season
merged = potential_skill.merge(an_bn, left_on=['region', 'model', 'season'], right_on=['region', 'model', 'season'], how='left')
merged = merged.dropna()

# keep models with potential skill > 0.3 and an_agreement > 0.4 and bn_agreement > 0.4
merged = merged[(merged['potential_skill'] > 0.3)]
merged['region_season'] = merged['region'] + " | " + merged['season'] # compile region and season into one column, split by " | "
merged = merged.drop(columns = ['region', 'season'])

# use the region_season as columns with the models as rows, and the values being if the model is in the region_season
merged = merged.pivot(index='model', columns='region_season', values='potential_skill').reset_index()
merged.loc[:, merged.columns != 'model'] = merged.loc[:, merged.columns != 'model'].notna()
merged

C:\Users\edwin\AppData\Local\Temp\ipykernel_129752\2917172863.py:12: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[ True  True  True  True  True  True  True  True  True  True]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  merged.loc[:, merged.columns != 'model'] = merged.loc[:, merged.columns != 'model'].notna()
C:\Users\edwin\AppData\Local\Temp\ipykernel_129752\2917172863.py:12: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[False False  True False  True False False False False False]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  merged.loc[:, merged.columns != 'model'] = merged.loc[:, merged.columns != 'model'].notna()


region_season,model,eastern_east_africa | OND,lake_victoria_basin | SON
0,CESM1,True,False
1,CMCC,True,False
2,CanESM5,True,True
3,DWD,True,False
4,ECMWF,True,True
5,GEM5,True,False
6,GFDL,True,False
7,JMA,True,False
8,METEO,True,False
9,NASA,True,False


'OND'

In [70]:
temp = temp.drop(columns=['corr', 'skill_score', 'conditional_bias', 'unconditional_bias'])
temp

,region,model,season,month_of_prediction,potential_skill
0,eastern_east_africa,CCSM4,MAM,1.0,0.067737
1,eastern_east_africa,CCSM4,MAM,2.0,0.140350
2,eastern_east_africa,CCSM4,MAM,3.0,0.377000
11,eastern_east_africa,CCSM4,OND,8.0,0.063975
12,eastern_east_africa,CCSM4,OND,9.0,0.285253
...,...,...,...,...,...
1281,west_africa,NASA,JAS,6.0,0.155863
1282,west_africa,NASA,JAS,7.0,0.204949
1287,west_africa,NCEP,JAS,5.0,0.155285
1288,west_africa,NCEP,JAS,6.0,0.211303


In [8]:
# Create a combined column for the region-season pair
stat_clean['region_season'] = stat_clean['region'] + " | " + stat_clean['season']

def draw_heatmap(*args, **kwargs):
    data = kwargs.pop('data')
    # Pivot so that x-axis is month_of_prediction and y-axis is model
    d = data.pivot(index='model', columns='month_of_prediction', values='conditional_bias')

    # Define mapping of season to its first month
    season_to_first = {
        "MAM": 3,
        "AMJ": 4,
        "MJJ": 5,
        "FMA": 2
    }

    # Get the season for this facet (assumes all rows share the same season)
    season_val = data['season'].iloc[0]
    first_month = season_to_first.get(season_val)
    if first_month is not None:
        # Create a list of 7 months ending with the season's first month
        desired_order = [ ((first_month - 6 + i - 1) % 12) + 1 for i in range(7) ]
        # Filter to only the months present in the pivot
        new_order = [m for m in desired_order if m in d.columns]
        if new_order:
            d = d[new_order]

    sns.heatmap(d,
                vmin=0, vmax=1,
                cmap=sns.color_palette('Blues', 10),
                fmt=".2f",
                linewidths=0.1, linecolor='black',
                square=True)
    plt.xticks(fontsize=7)
    plt.yticks(fontsize=7)
    plt.gca().invert_xaxis()

# Facet using the combined region_season column with col_wrap to make layout cleaner.
fg = sns.FacetGrid(stat_clean, col='region_season', sharex=False, sharey=True, col_wrap=4)
fg.map_dataframe(draw_heatmap)
fg.set_titles("{col_name}")
fg.set_ylabels("Model")
fg.set_xlabels("Month Of Prediction")

# moving the title to the top of the figure
fg.fig.subplots_adjust(top=0.9)
fg.fig.suptitle('Conditional Bias by Model and Month of Prediction', y=0.98)

plt.savefig('figures/seasonal_metrics/conditional_bias.png')
plt.close()

In [9]:
# Create a combined column for the region-season pair
stat_clean['region_season'] = stat_clean['region'] + " | " + stat_clean['season']

def draw_heatmap(*args, **kwargs):
    data = kwargs.pop('data')
    # Pivot so that x-axis is month_of_prediction and y-axis is model
    d = data.pivot(index='model', columns='month_of_prediction', values='unconditional_bias')

    # Define mapping of season to its first month
    season_to_first = {
        "MAM": 3,
        "AMJ": 4,
        "MJJ": 5,
        "FMA": 2
    }

    # Get the season for this facet (assumes all rows share the same season)
    season_val = data['season'].iloc[0]
    first_month = season_to_first.get(season_val)
    if first_month is not None:
        # Create a list of 7 months ending with the season's first month
        desired_order = [ ((first_month - 6 + i - 1) % 12) + 1 for i in range(7) ]
        # Filter to only the months present in the pivot
        new_order = [m for m in desired_order if m in d.columns]
        if new_order:
            d = d[new_order]

    sns.heatmap(d,
                cmap=sns.color_palette('Blues', 10),
                fmt=".2f",
                linewidths=0.1, linecolor='black',
                square=True)
    plt.xticks(fontsize=7)
    plt.yticks(fontsize=7)
    plt.gca().invert_xaxis()

# Facet using the combined region_season column with col_wrap to make layout cleaner.
fg = sns.FacetGrid(stat_clean, col='region_season', sharex=False, sharey=True, col_wrap=4)
fg.map_dataframe(draw_heatmap)
fg.set_titles("{col_name}")
fg.set_ylabels("Model")
fg.set_xlabels("Month Of Prediction")

# moving the title to the top of the figure
fg.fig.subplots_adjust(top=0.9)
fg.fig.suptitle('Unconditional Bias by Model and Month of Prediction', y=0.98)

plt.savefig('figures/seasonal_metrics/unconditional_bias.png')
plt.close()